In [ ]:
!nvidia-smi

In [ ]:
%%writefile qk_attn.cu
#include <stdio.h>
#include <cuda_runtime.h>
#include <math.h>

__global__ void qkScale(float *Q, float *K, float *S, int seq, int d) {
    int row = blockIdx.y*blockDim.y + threadIdx.y;
    int col = blockIdx.x*blockDim.x + threadIdx.x;
    if (row >= seq || col >= seq) return;
    float acc = 0.f;
    for (int i = 0; i < d; i++) acc += Q[row*d+i] * K[col*d+i];
    S[row*seq+col] = acc / sqrtf((float)d);
}

int main() {
    int seq = 512, d = 64;
    float *Q, *K, *S;
    cudaMalloc(&Q, seq*d*4); cudaMalloc(&K, seq*d*4); cudaMalloc(&S, seq*seq*4);
    dim3 blk(16,16), grd((seq+15)/16, (seq+15)/16);
    qkScale<<<grd,blk>>>(Q,K,S,seq,d); cudaDeviceSynchronize();
    cudaEvent_t s, e; cudaEventCreate(&s); cudaEventCreate(&e);
    int reps = 100; cudaEventRecord(s);
    for (int i = 0; i < reps; i++) qkScale<<<grd,blk>>>(Q,K,S,seq,d);
    cudaEventRecord(e); cudaEventSynchronize(e);
    float ms; cudaEventElapsedTime(&ms,s,e);
    printf("QK^T seq=%d d=%d  %.2f TFLOPS  %.2f ms\n",
           seq, d, 2.0*seq*seq*d*reps/(ms*1e9), ms/reps);
    cudaFree(Q); cudaFree(K); cudaFree(S);
}

In [ ]:
!nvcc -arch=sm_75 -O2 -o qk_attn qk_attn.cu && ./qk_attn